In [13]:
import os
import pandas as pd
import numpy as np
import xarray as xr
from scipy.spatial import cKDTree

# Define the path to the main data folder
data_folder = "data/FLUXNET_Data_Shuttle/"

# === Load metadata with KG info ===
metadata = pd.read_csv('data/FLUXNET_Data_Shuttle/complete_fluxnet_data_shuttle_metadata_table.csv')
metadata.rename(columns={'Site ID': 'Site'}, inplace=True)

# Keep only sites within ±51.6° latitude for the ISS range
metadata = metadata[
    (metadata['Lat'] <= 51.6) & (metadata['Lat'] >= -51.6) & (metadata['last_year'] >= 2018)
]

# Extract valid site IDs
valid_sites = set(metadata['Site'])

print(f"Number of sites within ±51.6°: {len(valid_sites)}")

nc_path_drought = "data/Support/spei01.nc"
ds_drought = xr.open_dataset(nc_path_drought, decode_times=True)
lat_drought = ds_drought['lat'].values
lon_drought = ds_drought['lon'].values
time_drought = pd.to_datetime(ds_drought['time'].values)
spei_values = ds_drought['spei'].values
spei_lookup = {
    (t.year, t.month): i
    for i, t in enumerate(time_drought)
}
lat2d_d, lon2d_d = np.meshgrid(lat_drought, lon_drought, indexing='ij')
points_drought = np.column_stack((
    lat2d_d.ravel(),
    lon2d_d.ravel()
))
tree_drought = cKDTree(points_drought)

Number of sites within ±51.6°: 462


In [14]:
# === Assign season ===
def assign_season(month, latitude):
    if pd.isna(latitude):
        return 'Other'
    if latitude >= 0:
        return 'Winter' if month in [12, 1, 2] else 'Summer' if month in [6, 7, 8] else 'Other'
    else:
        return 'Winter' if month in [6, 7, 8] else 'Summer' if month in [12, 1, 2] else 'Other'
    
def extract_spei(lat, lon, timestamp):
    timestamp = pd.to_datetime(timestamp)

    # Get matching SPEI month
    key = (timestamp.year, timestamp.month)
    time_idx = spei_lookup.get(key, None)
    if time_idx is None:
        return np.nan

    # Find nearest grid cell
    dist, idx = tree_drought.query((lat, lon))

    lat_idx, lon_idx = np.unravel_index(idx, (len(lat_drought), len(lon_drought)))

    val = spei_values[time_idx, lat_idx, lon_idx]

    if np.isnan(val) or np.isinf(val):
        return np.nan

    return val

In [15]:
print(os.listdir(data_folder))

['AMF_US-PFh_FLUXNET_2019-2019_v1.3_r1', 'AMF_US-Akn_FLUXNET_2011-2022_v1.3_r1', 'AMF_US-Rms_FLUXNET_2014-2023_v1.3_r1', 'AMF_US-CF1_FLUXNET_2017-2021_v1.3_r1', 'AMF_US-MN1_FLUXNET_2017-2024_v1.3_r1', 'TERN_AU-Cpr_FLUXNET_2010-2025_v1.3_r1', 'AMF_US-WT1_FLUXNET_2023-2024_v1.3_r1', 'AMF_CR-Fsc_FLUXNET_2015-2018_v1.3_r1', 'TERN_AU-Stp_FLUXNET_2008-2025_v1.3_r1', 'AMF_US-ChR_FLUXNET_2006-2019_v1.3_r1', 'FLX_IT-SR2_FLUXNET_2013-2024_v1.3_r1', 'AMF_US-SP1_FLUXNET_2000-2020_v1.3_r1', 'EUF_ES-Crg_FLUXNET_2021-2024_v1.3_r1', 'ICOS_IT-TrF_FLUXNET_2012-2024_v1.3_r1', 'AMF_AR-CCg_FLUXNET_2018-2024_v1.3_r1', 'ICOS_FR-FBn_FLUXNET_2008-2024_v1.3_r1', 'AMF_US-xJR_FLUXNET_2019-2024_v1.3_r1', 'EUF_CH-Rh2_FLUXNET_2018-2021_v1.3_r1', 'ICOS_FR-Mej_FLUXNET_2019-2024_v1.3_r1', 'EUF_AT-Neu_FLUXNET_2002-2020_v1.3_r1', 'AMF_US-Hn3_FLUXNET_2017-2018_v1.3_r1', 'AMF_US-Ne1_FLUXNET_2001-2024_v1.3_r1', 'AMF_US-Tw1_FLUXNET_2011-2023_v1.3_r1', 'AMF_US-NC2_FLUXNET_2005-2025_v1.3_r1', 'KOF_KR-UAO_FLUXNET_2017-2019_v1.3

In [ ]:
# Process daily data
file_pattern="FLUXMET_DD"
output_file=data_folder+"WUE_combined_daily_shuttle.csv"

all_data = []

# Loop through each site folder
for site_folder in os.listdir(data_folder):
    site_path = os.path.join(data_folder, site_folder)

    # Check if it's a folder
    if os.path.isdir(site_path):
        # Get site name and dates from the folder name
        folder_parts = site_folder.split("_")
        site_name = folder_parts[1]
        date_range = folder_parts[3]

        # Skip anything not in latitude range
        if site_name not in valid_sites:
            continue
        
        # Look for the desired file
        for file_name in os.listdir(site_path):
            if f"{site_name}_FLUXNET_{file_pattern}_{date_range}" in file_name and file_name.endswith(".csv"):
                file_path = os.path.join(site_path, file_name)

                # Read the data
                df = pd.read_csv(file_path)

                # Ensure required columns exist
                if 'LE_CORR' in df.columns and 'GPP_DT_VUT_USTAR50' in df.columns:
                    # Calculate WUE and keep relevant columns
                    if 'NEE_VUT_REF_QC' not in df.columns:
                        df['NEE_VUT_REF_QC'] = df['NEE_CUT_REF_QC']  # Not sure what this is 
                    df_filtered = df[(df['GPP_DT_VUT_USTAR50'] != -9999) & (df['LE_F_MDS_QC']>0.8) & (df['NEE_VUT_REF_QC']>0.8) & (df['LE_CORR'] > 5)]
                    if df_filtered.empty:
                        print(f"No valid data found in {file_name}. Skipping this file.")
                        continue
                    
                    # Convert LE_CORR (W m⁻²) to kg H₂O m⁻² s⁻¹
                    df_filtered['LE_CORR_kg_H2O'] = df_filtered['LE_CORR'] / (2501 - (2.36 * df_filtered['TA_F']))  # in J/g, latent heat adjusted by air temperature

                    # Calculate WUE in gC per kg H₂O
                    df_filtered['WUE'] = df_filtered['GPP_DT_VUT_USTAR50']*12.011*10**-6*1800 / (df_filtered['LE_CORR']/((2.501-0.00237*df_filtered['TA_F'])*10**6)*1800)
                    
                    # Add site information and retain necessary columns
                    df_filtered['Site'] = site_name

                    df_filtered['TIMESTAMP'] = pd.to_datetime(df_filtered['TIMESTAMP'], format='%Y%m%d')
                    df_filtered = df_filtered[df_filtered['TIMESTAMP'].dt.year >= 2018]
                    df_filtered['Month'] = df_filtered['TIMESTAMP'].dt.month

                    if df_filtered.empty:
                        print(f"No valid data after filtering in {file_name}. Skipping this file.")
                        continue
                    # Group by month and sum GPP to find the peak month
                    monthly_gpp = df_filtered.groupby('Month')['GPP_DT_VUT_USTAR50'].sum()
                    peak_month = monthly_gpp.idxmax()

                    # Get growing season months (circular across year boundaries)
                    growing_months = [(peak_month - 1) % 12 or 12, peak_month, (peak_month + 1 - 1) % 12 + 1]

                    # Create a new column indicating growing season
                    df_filtered['GrowingSeason'] = df_filtered['Month'].apply(lambda x: 'Growing' if x in growing_months else 'Non-Growing')

                    df_output = df_filtered[['TIMESTAMP', 'WUE','GPP_DT_VUT_USTAR50','LE_CORR_kg_H2O', 'Site', 'TA_F','GrowingSeason']]
                    # Append to the list
                    all_data.append(df_output)
                    print(f"Processed file: {file_path}")
                else:
                    print(f"Required columns not found in {file_name}. Skipping this file.")

                break  # Exit file loop once the desired file is found

# Combine all data into a single DataFrame
final_df = pd.concat(all_data, ignore_index=True)

final_df['TIMESTAMP'] = pd.to_datetime(final_df['TIMESTAMP'])
final_df = final_df.merge(
    metadata[['Site', 'Lat', 'Long','Veg', 'kg_class', 'kg_label']], 
    on='Site', how='left'
)

final_df['Season'] = final_df.apply(
    lambda row: assign_season(row['TIMESTAMP'].month, row['Lat']), axis=1
)
final_df = final_df[final_df['WUE'] != -9999].dropna(subset=['WUE'])

final_df['SPEI'] = final_df.apply(
    lambda row: extract_spei(row['Lat'], row['Long'], row['TIMESTAMP']), axis=1
)

# Save the combined data to a CSV file
final_df.to_csv(output_file, index=False)
print(f"Data processing complete. Combined data saved to {output_file}")

successful_sites = set(final_df['Site'])
metadata_success = metadata[metadata['Site'].isin(successful_sites)]
metadata_success.to_csv(data_folder+"FLUXNET_metadata_successful_daily_sites.csv", index=False)
final_df = []

In [ ]:
import os
import pandas as pd

# Define the path to the main data folder
data_folder = "data/FLUXNET_Data_Shuttle/"

# Process daily data
file_pattern="FLUXMET_HH"
output_file=data_folder+"WUE_combined_halfhourly_shuttle.csv"

all_data = []

# Loop through each site folder
for site_folder in os.listdir(data_folder):
    site_path = os.path.join(data_folder, site_folder)

    # Check if it's a folder
    if os.path.isdir(site_path):
        # Get site name and dates from the folder name
        folder_parts = site_folder.split("_")
        site_name = folder_parts[1]
        date_range = folder_parts[3]

        bif = pd.read_csv(f"{site_path}/{folder_parts[0]}_{folder_parts[1]}_FLUXNET_BIF_{folder_parts[3]}_{folder_parts[4]}_{folder_parts[5]}.csv")
        utc_offset = float(bif.loc[bif['VARIABLE']=='UTC_OFFSET', 'DATAVALUE'].values[0])
        
        # Skip anything not in latitude range
        if site_name not in valid_sites:
            continue

        # Look for the desired file
        for file_name in os.listdir(site_path):
            if f"{site_name}_FLUXNET_{file_pattern}_{date_range}" in file_name and file_name.endswith(".csv"):
                file_path = os.path.join(site_path, file_name)

                # Read the data
                df = pd.read_csv(file_path)

                # Ensure required columns exist
                if 'LE_CORR' in df.columns and 'GPP_DT_VUT_USTAR50' in df.columns:
                    # Calculate WUE and keep relevant columns
                    if 'NEE_VUT_REF_QC' not in df.columns:
                        df['NEE_VUT_REF_QC'] = df['NEE_CUT_REF_QC']  # Not sure what this is 
                    df_filtered = df[(df['GPP_DT_VUT_USTAR50'] != -9999) & (df['NEE_VUT_REF_QC']<2) & (df['LE_F_MDS_QC']<2) & (df['LE_CORR'] > 5)]
                    
                    # Convert LE_CORR (W m⁻²) to kg H₂O m⁻² s⁻¹
                    df_filtered['LE_CORR_kg_H2O'] = df_filtered['LE_CORR'] / (2501 - (2.36 * df_filtered['TA_F']))  # in J/g, latent heat adjusted by air temperature

                    # Calculate WUE in gC per kg H₂O
                    df_filtered['WUE'] = df_filtered['GPP_DT_VUT_USTAR50']*12.011*10**-6*1800 / (df_filtered['LE_CORR']/((2.501-0.00237*df_filtered['TA_F'])*10**6)*1800)
                    
                    # Add site information and retain necessary columns
                    df_filtered['Site'] = site_name
                    df_filtered['TIMESTAMP'] = pd.to_datetime(df_filtered['TIMESTAMP_START'], format='%Y%m%d%H%M')
                    df_filtered['UTC_Timestamp'] = df_filtered['TIMESTAMP'] - pd.to_timedelta(utc_offset, unit='h')
                    df_filtered = df_filtered[df_filtered['TIMESTAMP'].dt.year >= 2018]
                    
                    if df_filtered.empty:
                        print(f"No valid data found in {file_name}. Skipping this file.")
                        continue

                    df_output = df_filtered[['TIMESTAMP', 'WUE', 'GPP_DT_VUT_USTAR50','LE_CORR', 'Site','UTC_Timestamp']]
                    # Append to the list
                    all_data.append(df_output)
                    print(f"Processed file: {file_path}")
                else:
                    print(f"Required columns not found in {file_name}. Skipping this file.")
                break  # Exit file loop once the desired file is found

# Combine all data into a single DataFrame
final_df = pd.concat(all_data, ignore_index=True)

final_df['TIMESTAMP'] = pd.to_datetime(final_df['TIMESTAMP'], format='%Y%m%d%H%M')
final_df = final_df.merge(
    metadata[['Site', 'Lat', 'Long', 'Veg', 'kg_class', 'kg_label']], 
    on='Site', how='left'
)

# Compute UTC offset (hours) and round to nearest half-hour
final_df['UTC_offset'] = final_df['Long'] / 15.0
final_df['Local_Time'] = final_df['UTC_Timestamp'] + pd.to_timedelta(final_df['UTC_offset'], unit='h')
final_df['Local_Time_30min'] = final_df['Local_Time'].dt.round('30min')

final_df['Season'] = final_df.apply(
    lambda row: assign_season(row['TIMESTAMP'].month, row['Lat']), axis=1
)
final_df['TOD'] = final_df['Local_Time_30min'].dt.hour.map(
    lambda x: 'Morning' if x in [7, 8, 9, 10] else 
              'Midday' if x in [11, 12, 13, 14] else 
              'Afternoon' if x in [15, 16, 17, 18] else 'Other'
)
final_df = final_df[final_df['WUE'] != -9999].dropna(subset=['WUE'])

final_df['SPEI'] = final_df.apply(
    lambda row: extract_spei(row['Lat'], row['Long'], row['TIMESTAMP']), axis=1
)

# Save the combined data to a CSV file
final_df.to_csv(output_file, index=False)
print(f"Data processing complete. Combined data saved to {output_file}")

successful_sites = set(final_df['Site'])
metadata_success = metadata[metadata['Site'].isin(successful_sites)]
metadata_success.to_csv(data_folder+"FLUXNET_metadata_successful_hh_sites.csv", index=False)